# Трамвайный прогноз · DS‑1 · обучение на Kaggle (LightGBM, CatBoost, Календарь РФ 2025)

**Включите Internet и нажмите Run All. GPU не нужен.** Архив автоматически
скачается с https://disk.yandex.ru/d/DiFwlfMOauxjBg. Код моделей уже находится
внутри ноутбука: подключать GitHub или вводить токены не нужно.

1. Создайте Kaggle Notebook и импортируйте этот `.ipynb`.
2. В настройках сессии включите **Internet**. Добавлять данные через Add Input
   не требуется: ноутбук скачает `dataset.zip` по указанной публичной ссылке
   (около 2,54 ГБ), проверит размер/хеш и извлечёт только метки и шаблон.
3. Оставьте **Accelerator: None / CPU** и выполните все ячейки.
4. Скачайте `submission.csv` для платформы и `report.json` для проверки качества.
   Не публикуйте ноутбук с входными данными и обученным артефактом.

Цель — **валидации/час**, а не фактическая заполненность салона.
Модели:
- **LightGBM Direct** (L1 / MAE оптимизация под WAPE числитель)
- **CatBoost Direct** (MAE оптимизация)
- **CalendarProfile** (адаптивный профиль с учетом производственного календаря РФ 2025: рабочая суббота 01.11, нерабочие 03–04.11 и 31.12)
- **Blend Ensemble** (взвешенное ансамблирование)
- Логика возобновления маршрута 50 с 15.11.2025
- Маршрут 5: нулевой cold start на 1464 строки, как требуется шаблоном.


## 1. Настройки
Ссылка уже указана. Режим `yandex` скачивает архив автоматически; `input` оставлен для ранее загруженных файлов.


In [ ]:
import os
import sys
import json
import csv
import hashlib
import subprocess
import tempfile
import zipfile
import importlib.metadata
from pathlib import Path

PROFILES_ONLY = False  # True: только baseline, без бустинга; другой набор кандидатов.
RUN_TESTS = True
DATA_SOURCE = 'yandex'  # 'input' — необязательный ручной вариант через Add Input.
YANDEX_URL = 'https://disk.yandex.ru/d/DiFwlfMOauxjBg'
DOWNLOAD_CACHE = '/tmp/mthack_yandex'
INPUT_ROOT = Path(os.environ.get('MTHACK_INPUT_DIR', '/kaggle/input'))
WORK = Path(os.environ.get('MTHACK_WORK_DIR', '/kaggle/working'))
INPUT_FILES = {
    'train': '',   # например /kaggle/input/my-private-data/labels_day_train.csv
    'test': '',    # labels_day_test.csv — фактические метки сентября–октября
    'sample': '',  # исходный test_submission.csv за ноябрь–декабрь
}
os.environ['OMP_NUM_THREADS'] = '2'
os.environ['OPENBLAS_NUM_THREADS'] = '2'
if sys.version_info < (3, 11):
    raise RuntimeError('Нужен Python 3.11+; выберите актуальную Python-среду Kaggle.')
WORK.mkdir(parents=True, exist_ok=True)
RUNTIME = Path(tempfile.mkdtemp(prefix='mthack_ds1_code_'))
EXTRACTED = Path(tempfile.mkdtemp(prefix='mthack_ds1_inputs_'))
OUT = WORK / 'ds1_results'
print('Python:', sys.version.split()[0])
if not PROFILES_ONLY:
    for package in ['numpy', 'scipy', 'scikit-learn', 'lightgbm', 'catboost']:
        try:
            print(package, importlib.metadata.version(package))
        except importlib.metadata.PackageNotFoundError:
            print(package, 'не установлен (будет использован fallback при необходимости)')
print('Результаты:', OUT)


## 2. Встроенный код модели
Ячейка разворачивает точную копию модулей DS‑1 во временной папке.
Ничего не скачивается. Исходные данные не входят в ноутбук.


In [ ]:
SOURCES = {'ml/forecast/core.py': '"""Strict readers and full-grid scoring of organizer hourly labels (MSK)."""\nfrom __future__ import annotations\n\nimport csv\nimport hashlib\nimport math\nfrom collections import defaultdict\nfrom datetime import date, timedelta\nfrom pathlib import Path\n\nKey = tuple[str, date, int]\nLabels = dict[Key, int]\n\nRUSSIAN_HOLIDAYS_2025 = frozenset({\n    date(2025, 1, 1), date(2025, 1, 2), date(2025, 1, 3), date(2025, 1, 4),\n    date(2025, 1, 5), date(2025, 1, 6), date(2025, 1, 7), date(2025, 1, 8),\n    date(2025, 2, 23), date(2025, 3, 8),\n    date(2025, 5, 1), date(2025, 5, 2), date(2025, 5, 8), date(2025, 5, 9),\n    date(2025, 6, 12), date(2025, 6, 13),\n    date(2025, 11, 3), date(2025, 11, 4),\n    date(2025, 12, 31),\n})\n\nRUSSIAN_WORKING_WEEKENDS_2025 = frozenset({\n    date(2025, 11, 1),\n})\n\n\ndef is_workday(d: date) -> bool:\n    if d in RUSSIAN_WORKING_WEEKENDS_2025:\n        return True\n    if d in RUSSIAN_HOLIDAYS_2025:\n        return False\n    return d.weekday() < 5\n\n\ndef effective_weekday(d: date) -> int:\n    if d in RUSSIAN_WORKING_WEEKENDS_2025:\n        return 4\n    if d in RUSSIAN_HOLIDAYS_2025:\n        return 6\n    return d.weekday()\n\n\ndef calendar_day_type(d: date) -> int:\n    if not is_workday(d):\n        return 3 if (effective_weekday(d) == 6 or d in RUSSIAN_HOLIDAYS_2025) else 2\n    if d in RUSSIAN_WORKING_WEEKENDS_2025:\n        return 4\n    if d.weekday() == 4:\n        return 1\n    return 0\n\n\ndef days(start: date, end: date):\n    while start <= end:\n        yield start\n        start += timedelta(days=1)\n\n\ndef grid(routes, start, end):\n    return [(str(r), d, h) for r in routes for d in days(start, end) for h in range(24)]\n\n\ndef parse_key(row) -> Key:\n    route = row[\'route\']\n    if not route or not route.isascii() or not route.isdecimal() or int(route) <= 0:\n        raise ValueError(f\'Invalid route: {route!r}\')\n    day = date.fromisoformat(row[\'date\'])\n    if day.isoformat() != row[\'date\']:\n        raise ValueError(\'Date must be YYYY-MM-DD\')\n    hour = int(row[\'hour\'])\n    if str(hour) != row[\'hour\'] or not 0 <= hour <= 23:\n        raise ValueError(f\'Invalid hour: {row["hour"]!r}\')\n    return str(int(route)), day, hour\n\n\ndef load_labels(paths) -> Labels:\n    result = {}\n    for path in paths:\n        with Path(path).open(encoding=\'utf-8-sig\', newline=\'\') as f:\n            reader = csv.DictReader(f, delimiter=\';\')\n            if reader.fieldnames != [\'route\', \'date\', \'hour\', \'boardings\']:\n                raise ValueError(f\'Unexpected label schema: {reader.fieldnames}\')\n            for row in reader:\n                if None in row or any(v is None for v in row.values()):\n                    raise ValueError(\'Malformed CSV row\')\n                key = parse_key(row)\n                value = int(row[\'boardings\'])\n                if value < 0 or str(value) != row[\'boardings\']:\n                    raise ValueError(\'Boardings must be nonnegative integers\')\n                if key in result:\n                    raise ValueError(f\'Duplicate label key across input files: {key}\')\n                result[key] = value\n    if not result:\n        raise ValueError(\'Labels are empty\')\n    return result\n\n\ndef rounded(value) -> int:\n    value = float(value)\n    if not math.isfinite(value) or abs(value) > 10**12:\n        raise ValueError(\'Prediction must be finite and within 1e12\')\n    return max(0, round(value))  # Same half-to-even rule as DS-2 baseline.\n\n\ndef sha256(path):\n    with Path(path).open(\'rb\') as f:\n        return hashlib.file_digest(f, \'sha256\').hexdigest()\n\n\ndef score(keys, truth, predictions, peak_hours=None):\n    """Score exactly the requested grid; absent label = 0 is an explicit assumption."""\n    if len(keys) != len(set(keys)) or set(keys) != set(predictions):\n        raise ValueError(\'Prediction keys must match the unique evaluation grid exactly\')\n    if set(truth) - set(keys):\n        raise ValueError(\'Truth contains keys outside evaluation grid\')\n    groups = defaultdict(lambda: {\'rows\': 0, \'actual_sum\': 0, \'absolute_error_sum\': 0,\n                                  \'pred_minus_actual\': 0, \'missing_label_rows\': 0})\n    for key in keys:\n        route, day, hour = key\n        actual, pred = truth.get(key, 0), rounded(predictions[key])\n        if actual < 0 or not math.isfinite(actual):\n            raise ValueError(\'Invalid truth\')\n        names = [\'all\', f\'route:{route}\', f\'month:{day:%Y-%m}\',\n                 \'hours:day_06_22\' if 6 <= hour < 23 else \'hours:night_23_05\',\n                 \'labels:observed\' if key in truth else \'labels:missing_assumed_zero\']\n        if peak_hours is not None:\n            names.append(\'hours:train_peaks\' if hour in peak_hours.get(route, ())\n                         else \'hours:other\')\n        for name in names:\n            m = groups[name]\n            m[\'rows\'] += 1\n            m[\'actual_sum\'] += actual\n            m[\'absolute_error_sum\'] += abs(pred - actual)\n            m[\'pred_minus_actual\'] += pred - actual\n            m[\'missing_label_rows\'] += key not in truth\n    if not groups[\'all\'][\'actual_sum\']:\n        raise ValueError(\'Global WAPE undefined: sum(y) must be positive\')\n    for m in groups.values():\n        m[\'wape\'] = m[\'absolute_error_sum\'] / m[\'actual_sum\'] if m[\'actual_sum\'] else None\n        m[\'wape_score\'] = max(0, 1 - m[\'wape\']) if m[\'wape\'] is not None else None\n    return dict(sorted(groups.items()))\n\n\ndef peak_hours(history, routes):\n    totals = defaultdict(int)\n    for (r, _, h), y in history.items():\n        totals[r, h] += y\n    return {r: sorted(range(24), key=lambda h: (-totals[r, h], h))[:4] for r in routes}\n', 'ml/forecast/models.py': '"""Models receive ONLY history through cutoff. Predict never accepts future truth."""\nfrom __future__ import annotations\n\nimport math\nfrom collections import defaultdict\nfrom datetime import date, timedelta\nfrom statistics import median, mean\n\nfrom .core import grid, rounded, is_workday, effective_weekday, calendar_day_type\n\n\nclass Profile:\n    def __init__(self, statistic=\'mean\', missing=\'zero\', shrink=0.2, window=None,\n                 use_calendar=False, route50_rule=False):\n        if statistic not in (\'mean\', \'median\') or missing not in (\'zero\', \'observed\'):\n            raise ValueError(\'Invalid profile configuration\')\n        self.statistic, self.missing, self.shrink, self.window = statistic, missing, shrink, window\n        self.use_calendar = use_calendar\n        self.route50_rule = route50_rule\n\n    def fit(self, history, start, cutoff):\n        if not history or any(not start <= k[1] <= cutoff for k in history):\n            raise ValueError(\'Training history crosses cutoff or is empty\')\n        self.cutoff = cutoff\n        self.routes = sorted({k[0] for k in history})\n        if self.window:\n            start = max(start, cutoff - timedelta(days=self.window - 1))\n        filtered = {k: v for k, v in history.items() if k[1] >= start}\n        local, hour, route = defaultdict(list), defaultdict(list), defaultdict(list)\n        keys = grid(self.routes, start, cutoff) if self.missing == \'zero\' else filtered\n        for r, d, h in keys:\n            y = filtered.get((r, d, h), 0)\n            w = effective_weekday(d) if self.use_calendar else d.weekday()\n            local[r, w, h].append(y)\n            hour[r, h].append(y)\n            route[r].append(y)\n        fn = mean if self.statistic == \'mean\' else median\n        self.local = {k: fn(v) for k, v in local.items()}\n        self.hour = {k: fn(v) for k, v in hour.items()}\n        self.route = {k: fn(v) for k, v in route.items()}\n        return self\n\n    def predict(self, keys):\n        if any(k[1] <= self.cutoff for k in keys):\n            raise ValueError(\'Prediction dates must be after cutoff\')\n        result = {}\n        for key in keys:\n            r, d, h = key\n            if self.route50_rule and r == \'50\' and not is_workday(d) and date(2025, 9, 6) <= d < date(2025, 11, 15):\n                result[key] = 0\n                continue\n            w = effective_weekday(d) if self.use_calendar else d.weekday()\n            parent = self.hour.get((r, h), self.route.get(r, 0))\n            local = self.local.get((r, w, h), parent)\n            result[key] = rounded((1 - self.shrink) * local + self.shrink * parent)\n        return result\n\n\nclass CalendarProfile:\n    """Optimized calendar profile: blends recent 56-day trend with full history on effective weekdays."""\n    def __init__(self, recent_weight=0.6, window=56, route50_rule=True):\n        self.recent_weight = recent_weight\n        self.window = window\n        self.route50_rule = route50_rule\n\n    def fit(self, history, start, cutoff):\n        self.cutoff = cutoff\n        self.full_profile = Profile(\'median\', shrink=0.0, use_calendar=True,\n                                    route50_rule=self.route50_rule).fit(history, start, cutoff)\n        self.recent_profile = Profile(\'median\', shrink=0.0, window=self.window,\n                                      use_calendar=True, route50_rule=self.route50_rule).fit(history, start, cutoff)\n        return self\n\n    def predict(self, keys):\n        p_full = self.full_profile.predict(keys)\n        p_recent = self.recent_profile.predict(keys)\n        w = self.recent_weight\n        return {k: rounded(w * p_recent[k] + (1 - w) * p_full[k]) for k in keys}\n\n\nclass Boosting:\n    def __init__(self, kind=\'hgb\', recursive=False, use_calendar=True, route50_rule=True):\n        self.kind = kind\n        self.recursive = recursive\n        self.use_calendar = use_calendar\n        self.route50_rule = route50_rule\n\n    def _extract_profiles(self, history, start, cutoff):\n        full_med, full_mean = defaultdict(list), defaultdict(list)\n        rec_med, rec_mean = defaultdict(list), defaultdict(list)\n        hour_med = defaultdict(list)\n        rec_start = max(start, cutoff - timedelta(days=55))\n\n        keys = grid(self.routes, start, cutoff)\n        for r, d, h in keys:\n            y = history.get((r, d, h), 0)\n            eff_w = effective_weekday(d) if self.use_calendar else d.weekday()\n            full_med[r, eff_w, h].append(y)\n            full_mean[r, eff_w, h].append(y)\n            hour_med[r, h].append(y)\n            if d >= rec_start:\n                rec_med[r, eff_w, h].append(y)\n                rec_mean[r, eff_w, h].append(y)\n\n        self.p_full_med = {k: median(v) for k, v in full_med.items()}\n        self.p_full_mean = {k: mean(v) for k, v in full_mean.items()}\n        self.p_rec_med = {k: median(v) for k, v in rec_med.items()}\n        self.p_rec_mean = {k: mean(v) for k, v in rec_mean.items()}\n        self.p_hour_med = {k: median(v) for k, v in hour_med.items()}\n\n    def features(self, keys, context=None):\n        import numpy as np\n        result = []\n        if self.recursive:\n            for r, d, h in keys:\n                t = d.timetuple().tm_yday\n                row = [self.route_codes[r], d.weekday(), h, d.month, t,\n                       int(d.weekday() >= 5), math.sin(2 * math.pi * t / 365),\n                       math.cos(2 * math.pi * t / 365)]\n                if context is not None:\n                    for lag in (1, 7):\n                        key = r, d - timedelta(days=lag), h\n                        row.extend([context.get(key, 0), int(key in context)])\n                result.append(row)\n            return np.asarray(result, dtype=float)\n\n        for r, d, h in keys:\n            t = d.timetuple().tm_yday\n            eff_w = effective_weekday(d) if self.use_calendar else d.weekday()\n            workday = int(is_workday(d)) if self.use_calendar else int(d.weekday() < 5)\n            dtype = calendar_day_type(d) if self.use_calendar else (2 if d.weekday() >= 5 else 0)\n\n            pf_med = getattr(self, \'p_full_med\', {}).get((r, eff_w, h), getattr(self, \'p_hour_med\', {}).get((r, h), 0))\n            pf_mean = getattr(self, \'p_full_mean\', {}).get((r, eff_w, h), pf_med)\n            pr_med = getattr(self, \'p_rec_med\', {}).get((r, eff_w, h), pf_med)\n            pr_mean = getattr(self, \'p_rec_mean\', {}).get((r, eff_w, h), pf_mean)\n            ph_med = getattr(self, \'p_hour_med\', {}).get((r, h), 0)\n            trend_ratio = pr_mean / (pf_mean + 1.0) if pf_mean > 0 else 1.0\n\n            is_r50_susp = int(self.route50_rule and r == \'50\' and not is_workday(d)\n                              and date(2025, 9, 6) <= d < date(2025, 11, 15))\n\n            row = [\n                h,\n                d.weekday(),\n                eff_w,\n                dtype,\n                workday,\n                d.month,\n                d.day,\n                t,\n                math.sin(2 * math.pi * h / 24),\n                math.cos(2 * math.pi * h / 24),\n                math.sin(2 * math.pi * t / 365),\n                math.cos(2 * math.pi * t / 365),\n                pf_med,\n                pf_mean,\n                pr_med,\n                pr_mean,\n                ph_med,\n                trend_ratio,\n                is_r50_susp,\n            ]\n            result.append(row)\n        return np.asarray(result, dtype=float)\n\n    def fit(self, history, start, cutoff):\n        import numpy as np\n        if not history or any(not start <= k[1] <= cutoff for k in history):\n            raise ValueError(\'Training history crosses cutoff or is empty\')\n        self.cutoff = cutoff\n        self.routes = sorted({k[0] for k in history})\n        self.route_codes = {r: i for i, r in enumerate(self.routes)}\n        self._extract_profiles(history, start, cutoff)\n        self.history = dict(history)\n\n        keys = grid(self.routes, start, cutoff)\n        if self.recursive:\n            from sklearn.ensemble import HistGradientBoostingRegressor\n            keys = [k for k in keys if k[1] >= start + timedelta(days=7)]\n            self.model = HistGradientBoostingRegressor(\n                loss=\'absolute_error\', learning_rate=.08, max_iter=150,\n                max_leaf_nodes=15, min_samples_leaf=30, l2_regularization=2,\n                categorical_features=[0, 1, 2, 3, 5], early_stopping=False, random_state=2025)\n            self.model.fit(self.features(keys, history), [history.get(k, 0) for k in keys])\n            return self\n\n        route_keys = defaultdict(list)\n        for k in keys:\n            route_keys[k[0]].append(k)\n\n        self.models = {}\n        for r in self.routes:\n            r_keys = route_keys[r]\n            X = self.features(r_keys)\n            y = np.array([history.get(k, 0) for k in r_keys], dtype=float)\n\n            if self.kind == \'lgb\':\n                try:\n                    import lightgbm as lgb\n                    model = lgb.LGBMRegressor(\n                        objective=\'regression_l1\', metric=\'mae\', learning_rate=0.03,\n                        n_estimators=300, num_leaves=15, min_child_samples=10,\n                        subsample=0.85, colsample_bytree=0.85, random_state=2025,\n                        verbose=-1, n_jobs=2\n                    )\n                    model.fit(X, y)\n                    self.models[r] = model\n                    continue\n                except (ImportError, Exception):\n                    pass\n\n            if self.kind == \'catboost\':\n                try:\n                    import catboost as cb\n                    model = cb.CatBoostRegressor(\n                        loss_function=\'MAE\', eval_metric=\'MAE\', iterations=300,\n                        learning_rate=0.04, depth=5, random_seed=2025, verbose=0,\n                        thread_count=2\n                    )\n                    model.fit(X, y)\n                    self.models[r] = model\n                    continue\n                except (ImportError, Exception):\n                    pass\n\n            if self.kind == \'blend\':\n                submodels = []\n                try:\n                    import lightgbm as lgb\n                    m1 = lgb.LGBMRegressor(\n                        objective=\'regression_l1\', metric=\'mae\', learning_rate=0.03,\n                        n_estimators=300, num_leaves=15, min_child_samples=10,\n                        subsample=0.85, colsample_bytree=0.85, random_state=2025,\n                        verbose=-1, n_jobs=2\n                    )\n                    m1.fit(X, y)\n                    submodels.append((\'lgb\', m1, 0.5))\n                except (ImportError, Exception):\n                    pass\n\n                try:\n                    import catboost as cb\n                    m2 = cb.CatBoostRegressor(\n                        loss_function=\'MAE\', eval_metric=\'MAE\', iterations=300,\n                        learning_rate=0.04, depth=5, random_seed=2025, verbose=0,\n                        thread_count=2\n                    )\n                    m2.fit(X, y)\n                    submodels.append((\'catboost\', m2, 0.5))\n                except (ImportError, Exception):\n                    pass\n\n                if not submodels:\n                    from sklearn.ensemble import HistGradientBoostingRegressor\n                    m3 = HistGradientBoostingRegressor(\n                        loss=\'absolute_error\', learning_rate=0.04, max_iter=200,\n                        max_leaf_nodes=15, min_samples_leaf=10, l2_regularization=1,\n                        early_stopping=False, random_state=2025\n                    )\n                    m3.fit(X, y)\n                    submodels.append((\'hgb\', m3, 1.0))\n                self.models[r] = submodels\n                continue\n\n            from sklearn.ensemble import HistGradientBoostingRegressor\n            model = HistGradientBoostingRegressor(\n                loss=\'absolute_error\', learning_rate=0.04, max_iter=200,\n                max_leaf_nodes=15, min_samples_leaf=10, l2_regularization=1,\n                early_stopping=False, random_state=2025\n            )\n            model.fit(X, y)\n            self.models[r] = model\n\n        return self\n\n    def _predict_route(self, r, X):\n        m = getattr(self, \'models\', {}).get(r)\n        if m is None:\n            if hasattr(self, \'model\'):\n                return self.model.predict(X)\n            return [0] * len(X)\n        if isinstance(m, list):\n            total_w = sum(w for _, _, w in m)\n            preds = sum(w * sub.predict(X) for _, sub, w in m) / total_w\n            return preds\n        return m.predict(X)\n\n    def predict(self, keys):\n        if any(k[1] <= self.cutoff for k in keys):\n            raise ValueError(\'Prediction dates must be after cutoff\')\n        known = [k for k in keys if k[0] in self.route_codes]\n        result = {k: 0 for k in keys}\n        if not known:\n            return result\n\n        if not self.recursive:\n            route_known = defaultdict(list)\n            for k in known:\n                route_known[k[0]].append(k)\n            for r, r_keys in route_known.items():\n                X = self.features(r_keys)\n                raw_preds = self._predict_route(r, X)\n                for k, p in zip(r_keys, raw_preds):\n                    if self.route50_rule and r == \'50\' and not is_workday(k[1]) and date(2025, 9, 6) <= k[1] < date(2025, 11, 15):\n                        result[k] = 0\n                    else:\n                        result[k] = rounded(p)\n            return result\n\n        context = dict(self.history)\n        day, end = self.cutoff + timedelta(days=1), max(k[1] for k in known)\n        wanted = set(known)\n        while day <= end:\n            batch = grid(self.routes, day, day)\n            predictions = self.model.predict(self.features(batch, context))\n            for key, value in zip(batch, predictions):\n                context[key] = rounded(value)\n                if key in wanted:\n                    result[key] = context[key]\n            day += timedelta(days=1)\n        return result\n\n\ndef candidates(include_boosting=True):\n    result = {\n        \'mean_zero\': lambda: Profile(\'mean\'),\n        \'median_zero\': lambda: Profile(\'median\'),\n        \'mean_observed\': lambda: Profile(\'mean\', \'observed\'),\n        \'median_observed\': lambda: Profile(\'median\', \'observed\'),\n        \'mean_56d\': lambda: Profile(\'mean\', window=56),\n        \'profile_calendar\': lambda: CalendarProfile(recent_weight=0.6, window=56, route50_rule=True),\n    }\n    if include_boosting:\n        result.update(\n            hgb_direct=lambda: Boosting(kind=\'hgb\'),\n            lgb_direct=lambda: Boosting(kind=\'lgb\'),\n            catboost_direct=lambda: Boosting(kind=\'catboost\'),\n            blend_ensemble=lambda: Boosting(kind=\'blend\'),\n            hgb_recursive=lambda: Boosting(recursive=True),\n        )\n    return result\n\n', 'ml/forecast/run.py': '"""One-command chronological backtest, model selection and exact submission."""\nfrom __future__ import annotations\n\nimport argparse\nimport hashlib\nimport importlib.metadata\nimport json\nimport os\nimport pickle\nimport platform\nimport time\nfrom datetime import date\nfrom pathlib import Path\n\n# A bounded CPU budget also avoids OpenMP oversubscription on desktops.\nos.environ.setdefault(\'OMP_NUM_THREADS\', \'2\')\nos.environ.setdefault(\'OPENBLAS_NUM_THREADS\', \'2\')\n\nfrom .core import grid, load_labels, peak_hours, score, sha256\nfrom .models import candidates\nfrom ml.submit.adapter import export_api, read_sample, validate, write_submission\n\nFIRST, FINAL_CUTOFF = date(2025, 1, 1), date(2025, 10, 31)\nFOLDS = [\n    (\'may_jun\', date(2025, 4, 30), date(2025, 5, 1), date(2025, 6, 30)),\n    (\'jul_aug\', date(2025, 6, 30), date(2025, 7, 1), date(2025, 8, 31)),\n    (\'sep_oct\', date(2025, 8, 31), date(2025, 9, 1), date(2025, 10, 31)),\n]\n\n\ndef dump(path, data):\n    Path(path).write_text(json.dumps(data, ensure_ascii=False, indent=2, allow_nan=False) + \'\\n\',\n                          encoding=\'utf-8\')\n\n\ndef evaluate_fold(values, cutoff, start, end, factory):\n    history = {k: v for k, v in values.items() if FIRST <= k[1] <= cutoff}\n    routes = sorted({k[0] for k in history}, key=int)\n    truth = {k: v for k, v in values.items() if start <= k[1] <= end}\n    if {k[0] for k in truth} - set(routes):\n        raise ValueError(\'Validation has new routes: define cold-start evaluation explicitly\')\n    keys = grid(routes, start, end)\n    t = time.perf_counter()\n    model = factory().fit(history, FIRST, cutoff)\n    predictions = model.predict(keys)\n    metrics = score(keys, truth, predictions, peak_hours(history, routes))\n    return {\'cutoff\': str(cutoff), \'from\': str(start), \'to_inclusive\': str(end),\n            \'routes\': routes, \'metrics\': metrics,\n            \'fit_predict_score_seconds\': round(time.perf_counter() - t, 3)}\n\n\ndef select_dev(folds, names):\n    """Select on May–August only; pool absolute errors, never average route scores."""\n    losses = {name: sum(folds[fold][name][\'metrics\'][\'all\'][\'absolute_error_sum\']\n                       for fold in (\'may_jun\', \'jul_aug\')) for name in names}\n    return min(losses, key=lambda n: (losses[n], n)), losses\n\n\ndef main(argv=None):\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument(\'--train\', required=True, type=Path)\n    parser.add_argument(\'--test\', required=True, type=Path)\n    parser.add_argument(\'--sample\', required=True, type=Path)\n    parser.add_argument(\'--out\', required=True, type=Path)\n    parser.add_argument(\'--profiles-only\', action=\'store_true\',\n                        help=\'Skip optional scikit-learn candidates explicitly\')\n    parser.add_argument(\'--synthetic\', action=\'store_true\',\n                        help=\'Mark artificial smoke data and outputs as NOT FOR SUBMISSION\')\n    args = parser.parse_args(argv)\n    # Fail on malformed input before training or creating any outputs.\n    _, sample_keys = read_sample(args.sample)\n    train, test = load_labels([args.train]), load_labels([args.test])\n    if any(not FIRST <= k[1] <= date(2025, 8, 31) for k in train):\n        raise ValueError(\'Train must be January–August 2025 only\')\n    if any(not date(2025, 9, 1) <= k[1] <= FINAL_CUTOFF for k in test):\n        raise ValueError(\'Test labels must be September–October 2025 only\')\n    values = load_labels([args.train, args.test])\n    # Missing hours are allowed, an entirely missing date is a data availability problem.\n    for a, b, data in ((FIRST, date(2025, 8, 31), train),\n                       (date(2025, 9, 1), FINAL_CUTOFF, test)):\n        if {k[1] for k in data} != {k[1] for k in grid([\'check\'], a, b)}:\n            raise ValueError(\'Labels have missing full dates; audit before running model selection\')\n    if {k[0] for k in values} - {k[0] for k in sample_keys}:\n        raise ValueError(\'Historical routes absent from sample\')\n    models = candidates(not args.profiles_only)\n    if not args.profiles_only:\n        import sklearn  # noqa: F401; fail early instead of silently skipping boosting.\n    args.out.mkdir(parents=True, exist_ok=True)\n    inputs = {name: {\'name\': path.name, \'sha256\': sha256(path)}\n              for name, path in [(\'train\', args.train), (\'test\', args.test), (\'sample\', args.sample)]}\n    code_files = sorted(Path(__file__).parent.glob(\'*.py\')) + [\n        Path(__file__).parents[1] / \'submit\' / \'adapter.py\']\n    code = {str(p.relative_to(Path(__file__).parents[2])): sha256(p) for p in code_files}\n    installed_libs = {}\n    if not args.profiles_only:\n        for pkg in (\'numpy\', \'scipy\', \'scikit-learn\', \'lightgbm\', \'catboost\'):\n            try:\n                installed_libs[pkg] = importlib.metadata.version(pkg)\n            except importlib.metadata.PackageNotFoundError:\n                pass\n    report = {\n        \'schema_version\': 1, \'inputs\': inputs, \'code_sha256\': code,\n        \'data_kind\': \'synthetic_not_for_submission\' if args.synthetic else \'organizer_labels\',\n        \'python_version\': platform.python_version(),\n        \'library_versions\': installed_libs,\n        \'source_mode\': \'as_of_2025_10_31\', \'timezone\': \'Europe/Moscow\',\n        \'missing_label_policy\': \'zero for scoring; compare zero-filled and observed-only profiles\',\n        \'rounding\': \'clip negatives to zero; Python round half-to-even\',\n        \'peak_definition\': \'four largest hour sums per route in each training window only\',\n        \'night_definition\': \'23:00–05:59 MSK diagnostic; not an operating-hours assertion\',\n        \'external_factors\': \'Russian 2025 production calendar, route 50 weekend resumption policy\',\n        \'parameters\': {\'profile_shrink\': .2, \'short_window_days\': 56, \'recent_weight\': 0.6,\n                       \'lgb\': {\'objective\': \'regression_l1\', \'learning_rate\': .03, \'n_estimators\': 300, \'num_leaves\': 15},\n                       \'catboost\': {\'loss_function\': \'MAE\', \'iterations\': 300, \'learning_rate\': .04, \'depth\': 5},\n                       \'calendar\': \'Russian 2025 production calendar with working Saturdays and state holidays\',\n                       \'route50_policy\': \'weekend repair 2025-09-06..2025-11-14; resumed 2025-11-15\'},\n        \'folds\': {}, \'selection\': {}, \'platform_score\': None,\n    }\n    for fold, cutoff, start, end in FOLDS[:2]:\n        report[\'folds\'][fold] = {}\n        for name, factory in models.items():\n            result = evaluate_fold(values, cutoff, start, end, factory)\n            report[\'folds\'][fold][name] = result\n            print(f\'{fold} {name}: WAPE-score={result["metrics"]["all"]["wape_score"]:.6f}\', flush=True)\n        dump(args.out / \'report.json\', report)\n    proposed, losses = select_dev(report[\'folds\'], models)\n    baseline, _ = select_dev(report[\'folds\'], [\'mean_zero\', \'median_zero\'])\n    report[\'selection\'].update(development_candidate=proposed, development_baseline=baseline,\n                               development_absolute_errors=losses)\n    # Preselected candidate plus controls and calendar profile see Sep–Oct.\n    report[\'folds\'][\'sep_oct\'] = {}\n    candidates_to_eval = dict.fromkeys([baseline, \'mean_zero\', \'median_zero\', \'profile_calendar\', proposed])\n    for name in candidates_to_eval:\n        result = evaluate_fold(values, *FOLDS[2][1:], models[name])\n        report[\'folds\'][\'sep_oct\'][name] = result\n        print(f\'sep_oct {name}: WAPE-score={result["metrics"]["all"]["wape_score"]:.6f}\', flush=True)\n    latest = report[\'folds\'][\'sep_oct\']\n    err = lambda name: latest[name][\'metrics\'][\'all\'][\'absolute_error_sum\']\n    final_baseline = min([\'mean_zero\', \'median_zero\'], key=lambda n: (err(n), n))\n    best_candidate = min([c for c in [proposed, \'profile_calendar\'] if c in latest], key=lambda n: (err(n), n))\n    selected = best_candidate if err(best_candidate) < err(final_baseline) else final_baseline\n    report[\'selection\'].update(selected=selected,\n        final_baseline=final_baseline,\n        rule=\'candidate from pooled May–August error; keep it only if better than both registered baselines in Sep–Oct\',\n        caveat=\'Sep–Oct is a validation gate used in selection, not an untouched test estimate\')\n    # Refit final model on all available history through Oct 31, independently of backtest fits.\n    model = models[selected]().fit(values, FIRST, FINAL_CUTOFF)\n    predictions = model.predict(sample_keys)\n    cold = sorted({k[0] for k in sample_keys} - {k[0] for k in values}, key=int)\n    report[\'cold_start\'] = {\'routes\': cold, \'policy\': \'zero; no evidence to calibrate unseen routes\',\n                            \'evaluated\': False, \'route5_rows_preserved\': True}\n    identity = json.dumps({\'inputs\': inputs, \'code\': code, \'selected\': selected,\n                           \'parameters\': report[\'parameters\'], \'libraries\': report[\'library_versions\'],\n                           \'python\': report[\'python_version\']}, sort_keys=True).encode()\n    version = f\'ds1-{selected}-\' + hashlib.sha256(identity).hexdigest()[:12]\n    if args.synthetic:\n        version = \'synthetic-NOT-FOR-SUBMISSION-\' + version\n    report[\'model_version\'] = version\n    # Local artifact only. Never load an untrusted pickle and never publish organizer-derived artifacts.\n    artifact_path = args.out / \'model.local.pkl\'\n    artifact_path.write_bytes(pickle.dumps(model, protocol=5))\n    output_path = args.out / \'submission.csv\'\n    report[\'submission\'] = write_submission(args.sample, predictions, output_path)\n    export_api(sample_keys, predictions, args.out / \'forecast_api.csv\', version)\n    report[\'outputs_sha256\'] = {p.name: sha256(p) for p in\n                                (artifact_path, output_path, args.out / \'forecast_api.csv\')}\n    if sha256(args.sample) != inputs[\'sample\'][\'sha256\']:\n        raise RuntimeError(\'Original sample changed during run\')\n    validate(args.sample, output_path)\n    dump(args.out / \'report.json\', report)\n    print(f\'Selected {version}; validated {len(sample_keys)} rows. Platform result unknown.\', flush=True)\n    return report\n\n\nif __name__ == \'__main__\':\n    main()\n', 'ml/forecast/prepare_inputs.py': '"""Extract only labels/sample/description from a local organizer ZIP, never raw events."""\nimport argparse\nimport json\nimport zipfile\nfrom pathlib import Path\n\nfrom .core import sha256\n\n\ndef extract(archive, output):\n    required = {\'labels_day_train.csv\', \'labels_day_test.csv\', \'test_submission.csv\'}\n    output = Path(output)\n    with zipfile.ZipFile(archive) as z:\n        members = {}\n        for info in z.infolist():\n            name = Path(info.filename).name\n            if name not in required and name.lower() != \'readme.md\':\n                continue\n            if name in members:\n                raise ValueError(f\'Ambiguous archive: duplicate {name}\')\n            if info.file_size > 20_000_000:\n                raise ValueError(f\'Unexpectedly large label/description file: {name}\')\n            members[name] = info\n        if not required <= members.keys():\n            raise ValueError(f\'Missing archive members: {sorted(required - members.keys())}\')\n        if any((output / name).exists() for name in members):\n            raise ValueError(\'Refuse to overwrite original inputs; use an empty directory\')\n        output.mkdir(parents=True, exist_ok=True)\n        for name, info in members.items():\n            (output / name).write_bytes(z.read(info))\n    manifest = {\'archive_name\': Path(archive).name,\n                \'source_url\': \'https://disk.yandex.ru/d/DiFwlfMOauxjBg\',\n                \'files_sha256\': {name: sha256(output / name) for name in sorted(members)}}\n    (output / \'inputs_manifest.json\').write_text(json.dumps(manifest, indent=2) + \'\\n\')\n    return manifest\n\n\nif __name__ == \'__main__\':\n    p = argparse.ArgumentParser(description=__doc__)\n    p.add_argument(\'--archive\', required=True, type=Path)\n    p.add_argument(\'--out\', required=True, type=Path)\n    a = p.parse_args()\n    print(json.dumps(extract(a.archive, a.out), indent=2))\n', 'ml/forecast/test_pipeline.py': '"""Self-contained synthetic contract/leakage tests; no organizer data required."""\nimport csv\nimport importlib.util\nimport tempfile\nimport unittest\nimport zipfile\nfrom datetime import date, timedelta\nfrom pathlib import Path\n\nfrom ml.forecast.core import grid, load_labels, rounded, score, sha256\nfrom ml.forecast.models import Boosting, Profile\nfrom ml.forecast.run import evaluate_fold, select_dev\nfrom ml.forecast.prepare_inputs import extract\nfrom ml.submit.adapter import export_api, read_sample, validate, write_submission\n\n\nclass PipelineTests(unittest.TestCase):\n    def setUp(self):\n        self.tmp = tempfile.TemporaryDirectory()\n        self.root = Path(self.tmp.name)\n\n    def tearDown(self):\n        self.tmp.cleanup()\n\n    def sample(self):\n        path = self.root / \'sample.csv\'\n        # Deliberately reverse order to detect implicit model sorting of sample rows.\n        keys = grid([str(i) for i in range(1, 11)], date(2025, 11, 1), date(2025, 12, 31))[::-1]\n        with path.open(\'w\', newline=\'\') as f:\n            writer = csv.writer(f, delimiter=\';\')\n            writer.writerow([\'route\', \'date\', \'hour\', \'prediction\'])\n            writer.writerows((r, d, h, \'\') for r, d, h in keys)\n        return path, keys\n\n    def test_exact_sample_roundtrip_api_sums_and_route5(self):\n        sample, keys = self.sample()\n        digest = sha256(sample)\n        preds = {k: (int(k[0]) * 10 + k[2] + .5) for k in keys}\n        out = self.root / \'out.csv\'\n        result = write_submission(sample, preds, out)\n        self.assertEqual(result[\'route5_rows\'], 1464)\n        self.assertEqual(read_sample(out)[1], keys)\n        self.assertEqual(sha256(sample), digest)\n        api = self.root / \'api.csv\'\n        export_api(keys, preds, api, \'synthetic-test\')\n        with api.open() as f:\n            rows = list(csv.DictReader(f))\n        self.assertEqual(sum(int(r[\'pred_validations\']) for r in rows), result[\'prediction_sum\'])\n        self.assertTrue(all(r[\'timestamp_msk\'].endswith(\'+03:00\') for r in rows))\n        daily = {}\n        monthly = {}\n        for row in rows:\n            d = row[\'timestamp_msk\'][:10]\n            daily[d] = daily.get(d, 0) + int(row[\'pred_validations\'])\n        for d, value in daily.items():\n            monthly[d[:7]] = monthly.get(d[:7], 0) + value\n        self.assertEqual(sum(monthly.values()), result[\'prediction_sum\'])\n\n    def test_reject_bad_submission_inputs(self):\n        sample, keys = self.sample()\n        predictions = dict.fromkeys(keys, 1)\n        out = self.root / \'out.csv\'\n        with self.assertRaises(ValueError):\n            write_submission(sample, predictions, sample)\n        predictions.pop(keys[0])\n        with self.assertRaises(ValueError):\n            write_submission(sample, predictions, out)\n        predictions[keys[0]] = float(\'nan\')\n        with self.assertRaises(ValueError):\n            write_submission(sample, predictions, out)\n        predictions[keys[0]] = 1\n        write_submission(sample, predictions, out)\n        lines = out.read_text().splitlines()\n        lines[1], lines[2] = lines[2], lines[1]\n        out.write_text(\'\\n\'.join(lines) + \'\\n\')\n        with self.assertRaises(ValueError):\n            validate(sample, out)\n\n    def test_sample_missing_duplicate_wrong_period(self):\n        path, _ = self.sample()\n        lines = path.read_text().splitlines()\n        path.write_text(\'\\n\'.join(lines[:-1]) + \'\\n\')\n        with self.assertRaises(ValueError):\n            read_sample(path)\n        path.write_text(\'\\n\'.join(lines[:-1] + [lines[1]]) + \'\\n\')\n        with self.assertRaises(ValueError):\n            read_sample(path)\n        path.write_text(\'\\n\'.join(lines).replace(\'2025-11-01\', \'2025-10-31\') + \'\\n\')\n        with self.assertRaises(ValueError):\n            read_sample(path)\n\n    def test_metric_is_pooled_and_includes_missing_hours(self):\n        keys = [(\'1\', date(2025, 9, 1), 7), (\'2\', date(2025, 9, 1), 7)]\n        m = score(keys, {keys[0]: 100}, {keys[0]: 80, keys[1]: 10})\n        self.assertAlmostEqual(m[\'all\'][\'wape_score\'], .7)\n        self.assertIsNone(m[\'route:2\'][\'wape_score\'])\n        with self.assertRaises(ValueError):\n            score(keys, {}, dict.fromkeys(keys, 0))\n        with self.assertRaises(ValueError):\n            score(keys, {keys[0]: 1}, {keys[0]: 1})\n\n    def test_reader_rejects_duplicate_files_and_negative_counts(self):\n        p = self.root / \'labels.csv\'\n        p.write_text(\'route;date;hour;boardings\\n7;2025-01-01;0;2\\n\')\n        with self.assertRaises(ValueError):\n            load_labels([p, p])\n        p.write_text(\'route;date;hour;boardings\\n7;2025-01-01;0;-2\\n\')\n        with self.assertRaises(ValueError):\n            load_labels([p])\n\n    def test_fit_rejects_future_and_predict_rejects_past(self):\n        cutoff = date(2025, 1, 31)\n        history = {(\'7\', date(2025, 1, 1), 7): 20}\n        model = Profile().fit(history, date(2025, 1, 1), cutoff)\n        with self.assertRaises(ValueError):\n            model.predict(list(history))\n        history[\'7\', cutoff + timedelta(days=1), 7] = 1000\n        with self.assertRaises(ValueError):\n            Profile().fit(history, date(2025, 1, 1), cutoff)\n\n    def test_backtest_never_passes_future_labels_to_model(self):\n        values = {(\'7\', date(2025, 1, 1), 7): 10,\n                  (\'7\', date(2025, 2, 1), 7): 2000}\n        class Spy(Profile):\n            def fit(self, history, start, cutoff):\n                assert max(k[1] for k in history) <= cutoff\n                assert max(history.values()) == 10\n                return super().fit(history, start, cutoff)\n        result = evaluate_fold(values, date(2025, 1, 31), date(2025, 2, 1), date(2025, 2, 1), Spy)\n        self.assertEqual(result[\'metrics\'][\'all\'][\'rows\'], 24)\n\n    def test_zero_cold_start_and_missing_training_sensitivity(self):\n        history = {(\'7\', date(2025, 1, 1), 7): 40}\n        keys = [(\'7\', date(2025, 2, 5), 7), (\'5\', date(2025, 2, 5), 7)]\n        zero = Profile().fit(history, date(2025, 1, 1), date(2025, 1, 31)).predict(keys)\n        obs = Profile(missing=\'observed\').fit(history, date(2025, 1, 1), date(2025, 1, 31)).predict(keys)\n        self.assertEqual(zero[keys[1]], 0)\n        self.assertLess(zero[keys[0]], obs[keys[0]])\n\n    def test_rounding_is_finite_nonnegative(self):\n        self.assertEqual([rounded(x) for x in [-2, .5, 1.5, 2.5]], [0, 0, 2, 2])\n        for x in [float(\'inf\'), float(\'nan\'), 10**15]:\n            with self.assertRaises(ValueError):\n                rounded(x)\n\n    def test_selection_does_not_use_sep_oct(self):\n        def entry(n):\n            return {\'metrics\': {\'all\': {\'absolute_error_sum\': n}}}\n        folds = {\'may_jun\': {\'a\': entry(1), \'b\': entry(10)},\n                 \'jul_aug\': {\'a\': entry(5), \'b\': entry(1)},\n                 \'sep_oct\': {\'a\': entry(10000), \'b\': entry(0)}}\n        self.assertEqual(select_dev(folds, [\'a\', \'b\'])[0], \'a\')\n\n    def test_archive_extracts_only_needed_files_and_preserves_originals(self):\n        archive = self.root / \'data.zip\'\n        with zipfile.ZipFile(archive, \'w\') as z:\n            for name in [\'labels_day_train.csv\', \'labels_day_test.csv\', \'test_submission.csv\']:\n                z.writestr(\'nested/\' + name, \'synthetic\')\n            z.writestr(\'../../raw.csv\', \'must not be extracted\')\n            z.writestr(\'README.md\', \'Synthetic description\')\n        dest = self.root / \'inputs\'\n        manifest = extract(archive, dest)\n        self.assertEqual(len(manifest[\'files_sha256\']), 4)\n        self.assertFalse((self.root / \'raw.csv\').exists())\n        self.assertFalse((dest / \'raw.csv\').exists())\n        with self.assertRaises(ValueError):\n            extract(archive, dest)\n\n    @unittest.skipUnless(importlib.util.find_spec(\'numpy\'), \'optional boosting dependency\')\n    def test_recursive_lags_use_predictions_and_fill_intervening_days(self):\n        class FakeRegressor:\n            def __init__(self):\n                self.batches = []\n            def predict(self, features):\n                self.batches.append(features)\n                return [7] * len(features)\n        model = Boosting(recursive=True)\n        model.cutoff = date(2025, 10, 31)\n        model.routes, model.route_codes = [\'7\'], {\'7\': 0}\n        model.history = {(\'7\', model.cutoff, h): 99 for h in range(24)}\n        model.model = FakeRegressor()\n        key = (\'7\', date(2025, 11, 3), 12)\n        self.assertEqual(model.predict([key]), {key: 7})\n        self.assertEqual(len(model.model.batches), 3)\n        self.assertTrue(all(row[8] == 99 for row in model.model.batches[0]))\n        self.assertTrue(all(row[8] == 7 for row in model.model.batches[1]))\n        self.assertTrue(all(row[8] == 7 for row in model.model.batches[2]))\n        self.assertEqual(model.history[\'7\', model.cutoff, 12], 99)\n\n    def test_russian_calendar_2025(self):\n        from ml.forecast.core import is_workday, effective_weekday, calendar_day_type\n        # 2025-11-01 is a working Saturday\n        self.assertTrue(is_workday(date(2025, 11, 1)))\n        self.assertEqual(effective_weekday(date(2025, 11, 1)), 4)\n        self.assertEqual(calendar_day_type(date(2025, 11, 1)), 4)\n\n        # 2025-11-02 is Sunday\n        self.assertFalse(is_workday(date(2025, 11, 2)))\n        self.assertEqual(effective_weekday(date(2025, 11, 2)), 6)\n\n        # 2025-11-03 is non-working day (transferred holiday)\n        self.assertFalse(is_workday(date(2025, 11, 3)))\n        self.assertEqual(effective_weekday(date(2025, 11, 3)), 6)\n\n        # 2025-11-04 is National Unity Day\n        self.assertFalse(is_workday(date(2025, 11, 4)))\n        self.assertEqual(effective_weekday(date(2025, 11, 4)), 6)\n\n        # 2025-11-05 is normal Wednesday\n        self.assertTrue(is_workday(date(2025, 11, 5)))\n        self.assertEqual(effective_weekday(date(2025, 11, 5)), 2)\n\n        # 2025-12-31 is non-working day\n        self.assertFalse(is_workday(date(2025, 12, 31)))\n        self.assertEqual(effective_weekday(date(2025, 12, 31)), 6)\n\n    def test_calendar_profile_and_route50_rule(self):\n        from ml.forecast.models import CalendarProfile\n        history = {}\n        for d in [date(2025, 10, 1) + timedelta(days=i) for i in range(30)]:\n            for h in range(24):\n                history[\'50\', d, h] = 50 if d.weekday() < 5 else 0\n                history[\'1\', d, h] = 100 if d.weekday() < 5 else 20\n        cutoff = date(2025, 10, 31)\n        model = CalendarProfile(recent_weight=0.5, window=28, route50_rule=True).fit(history, date(2025, 10, 1), cutoff)\n\n        # 2025-11-01 is working Saturday: route 1 should have weekday traffic (~100)\n        p1_work = model.predict([(\'1\', date(2025, 11, 1), 12)])[(\'1\', date(2025, 11, 1), 12)]\n        self.assertGreater(p1_work, 80)\n\n        # 2025-11-04 is holiday: route 1 should have weekend/Sunday traffic (~20)\n        p1_hol = model.predict([(\'1\', date(2025, 11, 4), 12)])[(\'1\', date(2025, 11, 4), 12)]\n        self.assertLess(p1_hol, 30)\n\n        # Route 50 suspended on weekend before Nov 15\n        p50_susp = model.predict([(\'50\', date(2025, 11, 2), 12)])[(\'50\', date(2025, 11, 2), 12)]\n        self.assertEqual(p50_susp, 0)\n\n    @unittest.skipUnless(importlib.util.find_spec(\'lightgbm\'), \'lightgbm required\')\n    def test_lgbm_direct_fit_and_predict(self):\n        history = {}\n        for d in [date(2025, 9, 1) + timedelta(days=i) for i in range(30)]:\n            for h in range(24):\n                history[\'1\', d, h] = 100 + h * 2 if d.weekday() < 5 else 20 + h\n        cutoff = date(2025, 9, 30)\n        model = Boosting(kind=\'lgb\', recursive=False).fit(history, date(2025, 9, 1), cutoff)\n        preds = model.predict([(\'1\', date(2025, 10, 1), 10), (\'1\', date(2025, 10, 4), 10)])\n        self.assertGreater(preds[\'1\', date(2025, 10, 1), 10], preds[\'1\', date(2025, 10, 4), 10])\n\n    @unittest.skipUnless(importlib.util.find_spec(\'catboost\'), \'catboost required\')\n    def test_catboost_direct_fit_and_predict(self):\n        history = {}\n        for d in [date(2025, 9, 1) + timedelta(days=i) for i in range(30)]:\n            for h in range(24):\n                history[\'1\', d, h] = 100 + h * 2 if d.weekday() < 5 else 20 + h\n        cutoff = date(2025, 9, 30)\n        model = Boosting(kind=\'catboost\', recursive=False).fit(history, date(2025, 9, 1), cutoff)\n        preds = model.predict([(\'1\', date(2025, 10, 1), 10)])\n        self.assertGreater(preds[\'1\', date(2025, 10, 1), 10], 50)\n\n\nif __name__ == \'__main__\':\n    unittest.main()\n\n', 'ml/forecast/download_yandex.py': '"""Download organizer ZIP through the public Yandex Disk API, without credentials."""\nimport hashlib\nimport json\nimport re\nimport shutil\nimport time\nimport urllib.parse\nimport urllib.request\nimport zipfile\nfrom datetime import datetime, timezone\nfrom pathlib import Path\n\nPUBLIC_URL = \'https://disk.yandex.ru/d/DiFwlfMOauxjBg\'\nAPI = \'https://cloud-api.yandex.net/v1/disk/public/resources\'\n\n\ndef api_json(suffix, params, attempts=3):\n    url = API + suffix + \'?\' + urllib.parse.urlencode(params)\n    for attempt in range(attempts):\n        try:\n            with urllib.request.urlopen(url, timeout=30) as response:\n                return json.load(response)\n        except (OSError, ValueError) as exc:\n            if attempt == attempts - 1:\n                raise RuntimeError(\'API Яндекс Диска недоступен. Включите Internet в Kaggle; \'\n                                   \'если уже включён, повторите ячейку позже. \'\n                                   \'Также возможен лимит скачивания публичной ссылки.\') from exc\n            print(f\'API: повторная попытка {attempt + 2}/{attempts}\', flush=True)\n            time.sleep(2 ** (attempt + 1))\n\n\ndef digest(path, algorithm=\'sha256\'):\n    with Path(path).open(\'rb\') as f:\n        return hashlib.file_digest(f, algorithm).hexdigest()\n\n\ndef download(public_url=PUBLIC_URL, cache_dir=\'/tmp/mthack_yandex\', attempts=3):\n    params = {\'public_key\': public_url}\n    meta = api_json(\'\', params)\n    if meta.get(\'type\') == \'dir\':\n        # Organizer link is a public folder containing dataset.zip.\n        params[\'path\'] = \'/dataset.zip\'\n        meta = api_json(\'\', params)\n    if meta.get(\'type\') != \'file\' or not meta.get(\'name\', \'\').lower().endswith(\'.zip\'):\n        raise ValueError(\'Ссылка должна вести на ZIP или папку с dataset.zip\')\n    expected_size = int(meta[\'size\'])\n    if expected_size <= 0:\n        raise ValueError(\'Пустой архив в метаданных Яндекс Диска\')\n    algorithm = \'sha256\' if meta.get(\'sha256\') else \'md5\'\n    expected_hash = meta.get(algorithm)\n    identity = hashlib.sha256(json.dumps(params, sort_keys=True).encode()).hexdigest()[:16]\n    directory = Path(cache_dir) / identity\n    directory.mkdir(parents=True, exist_ok=True)\n    target, partial = directory / \'dataset.zip\', directory / \'dataset.zip.part\'\n\n    def valid(path):\n        return (path.exists() and path.stat().st_size == expected_size\n                and (not expected_hash or digest(path, algorithm) == expected_hash)\n                and zipfile.is_zipfile(path))\n\n    if valid(target):\n        print(\'Используется ранее проверенный архив:\', target, flush=True)\n        return target\n    print(f\'Архив: {expected_size / 1024**3:.2f} ГиБ. Потоковое скачивание во временную папку.\',\n          flush=True)\n    for attempt in range(attempts):\n        try:\n            offset = partial.stat().st_size if partial.exists() else 0\n            if offset >= expected_size:\n                if valid(partial):\n                    break\n                partial.unlink()\n                offset = 0\n            if shutil.disk_usage(directory).free < expected_size - offset + 64 * 1024**2:\n                raise RuntimeError(\'Недостаточно места для архива во временной папке\')\n            href = api_json(\'/download\', params)[\'href\']\n            if urllib.parse.urlparse(href).scheme != \'https\':\n                raise ValueError(\'API вернул незащищённую ссылку скачивания\')\n            headers = {\'Range\': f\'bytes={offset}-\'} if offset else {}\n            request = urllib.request.Request(href, headers=headers)\n            with urllib.request.urlopen(request, timeout=60) as response:\n                status = response.status\n                if status == 206:\n                    match = re.fullmatch(r\'bytes (\\d+)-(\\d+)/(\\d+)\',\n                                         response.headers.get(\'Content-Range\', \'\'))\n                    if not match or int(match[1]) != offset or int(match[3]) != expected_size:\n                        raise ValueError(\'Некорректный диапазон при докачке\')\n                    mode = \'ab\'\n                elif status == 200:\n                    offset, mode = 0, \'wb\'  # Server ignored Range: safely restart.\n                else:\n                    raise ValueError(f\'Неожиданный HTTP-статус: {status}\')\n                received, logged = offset, offset\n                with partial.open(mode) as f:\n                    while chunk := response.read(4 * 1024**2):\n                        f.write(chunk)\n                        received += len(chunk)\n                        if received > expected_size:\n                            raise ValueError(\'Размер загрузки превысил метаданные\')\n                        if received - logged >= 64 * 1024**2 or received == expected_size:\n                            print(f\'Скачано {received / expected_size:.1%} \'\n                                  f\'({received / 1024**2:.0f}/{expected_size / 1024**2:.0f} МиБ)\',\n                                  flush=True)\n                            logged = received\n            if partial.stat().st_size != expected_size:\n                raise OSError(\'Соединение прервано до конца файла\')\n            if not valid(partial):\n                partial.unlink()\n                raise ValueError(\'Проверка хеша или структуры ZIP не пройдена\')\n            break\n        except (OSError, ValueError) as exc:\n            if attempt == attempts - 1:\n                raise RuntimeError(\'Архив не скачан после повторных попыток. \'\n                                   \'Проверьте Internet и повторите ячейку: частичный файл сохранён \'\n                                   \'для докачки, если сервер поддерживает Range.\') from exc\n            print(f\'Скачивание: повторная попытка {attempt + 2}/{attempts}\', flush=True)\n            time.sleep(2 ** (attempt + 1))\n    partial.replace(target)\n    manifest = {\'source_url\': public_url, \'path\': params.get(\'path\'), \'size\': expected_size,\n                \'sha256\': digest(target), \'fetched_at_utc\': datetime.now(timezone.utc).isoformat(),\n                \'server_hash_algorithm\': algorithm if expected_hash else None,\n                \'server_hash_verified\': bool(expected_hash)}\n    (directory / \'download_manifest.json\').write_text(json.dumps(manifest, indent=2) + \'\\n\')\n    print(\'Архив скачан и проверен:\', target, flush=True)\n    return target\n', 'ml/forecast/test_download_yandex.py': '"""Network-independent tests: streaming, resume, cache and integrity checks."""\nimport hashlib\nimport io\nimport tempfile\nimport unittest\nimport zipfile\nfrom pathlib import Path\nfrom unittest.mock import patch\n\nfrom ml.forecast import download_yandex as yd\n\n\nclass Response(io.BytesIO):\n    def __init__(self, data, status=200, headers=None):\n        super().__init__(data)\n        self.status = status\n        self.headers = headers or {}\n\n\nclass DownloadTests(unittest.TestCase):\n    def setUp(self):\n        self.tmp = tempfile.TemporaryDirectory()\n        self.root = Path(self.tmp.name)\n        buffer = io.BytesIO()\n        with zipfile.ZipFile(buffer, \'w\') as z:\n            z.writestr(\'labels/synthetic.txt\', \'artificial fixture\')\n        self.blob = buffer.getvalue()\n        self.meta = {\'type\': \'file\', \'name\': \'dataset.zip\', \'size\': len(self.blob),\n                     \'sha256\': hashlib.sha256(self.blob).hexdigest()}\n\n    def tearDown(self):\n        self.tmp.cleanup()\n\n    def api(self, suffix, params):\n        if suffix == \'/download\':\n            self.assertEqual(params[\'path\'], \'/dataset.zip\')\n            return {\'href\': \'https://download.example.invalid/data.zip\'}\n        return self.meta if \'path\' in params else {\'type\': \'dir\'}\n\n    def run_download(self, opener, attempts=2):\n        with patch.object(yd, \'api_json\', side_effect=self.api), \\\n             patch.object(yd.urllib.request, \'urlopen\', side_effect=opener), \\\n             patch.object(yd.time, \'sleep\'):\n            return yd.download(cache_dir=self.root, attempts=attempts)\n\n    def test_folder_download_and_verified_cache(self):\n        target = self.run_download(lambda *a, **kw: Response(self.blob))\n        self.assertEqual(target.read_bytes(), self.blob)\n        with patch.object(yd, \'api_json\', side_effect=self.api), \\\n             patch.object(yd.urllib.request, \'urlopen\') as opened:\n            self.assertEqual(yd.download(cache_dir=self.root), target)\n            opened.assert_not_called()\n\n    def test_interruption_resumes_with_range(self):\n        blob = self.blob\n        class Interrupted(Response):\n            def read(self, size=-1):\n                if self.tell():\n                    raise OSError(\'simulated disconnect\')\n                return super().read(40)\n        calls = []\n        def open_response(request, **kwargs):\n            calls.append(request)\n            if len(calls) == 1:\n                return Interrupted(blob)\n            self.assertEqual(request.get_header(\'Range\'), \'bytes=40-\')\n            return Response(blob[40:], 206,\n                            {\'Content-Range\': f\'bytes 40-{len(blob)-1}/{len(blob)}\'})\n        self.assertEqual(self.run_download(open_response).read_bytes(), self.blob)\n        self.assertEqual(len(calls), 2)\n\n    def test_ignored_range_restarts_instead_of_appending(self):\n        class Interrupted(Response):\n            def read(self, size=-1):\n                if self.tell():\n                    raise OSError(\'disconnect\')\n                return super().read(40)\n        responses = iter([Interrupted(self.blob), Response(self.blob)])\n        self.assertEqual(self.run_download(lambda *a, **kw: next(responses)).read_bytes(), self.blob)\n\n    def test_wrong_hash_never_becomes_completed_archive(self):\n        self.meta[\'sha256\'] = \'0\' * 64\n        with self.assertRaises(RuntimeError):\n            self.run_download(lambda *a, **kw: Response(self.blob), attempts=1)\n        self.assertFalse(list(self.root.rglob(\'dataset.zip\')))\n\n    def test_wrong_range_never_becomes_completed_archive(self):\n        with self.assertRaises(RuntimeError):\n            self.run_download(lambda *a, **kw: Response(self.blob, 206,\n                              {\'Content-Range\': f\'bytes 5-{len(self.blob)-1}/{len(self.blob)}\'}),\n                              attempts=1)\n        self.assertFalse(list(self.root.rglob(\'dataset.zip\')))\n\n    def test_api_retries_and_reports_network_problem(self):\n        with patch.object(yd.urllib.request, \'urlopen\', side_effect=OSError(\'timeout\')) as opened, \\\n             patch.object(yd.time, \'sleep\'):\n            with self.assertRaisesRegex(RuntimeError, \'Internet\'):\n                yd.api_json(\'\', {\'public_key\': yd.PUBLIC_URL}, attempts=2)\n            self.assertEqual(opened.call_count, 2)\n\n\nif __name__ == \'__main__\':\n    unittest.main()\n', 'ml/submit/adapter.py': '"""Preserve the exact organizer sample keys, ordering and non-target text."""\nfrom __future__ import annotations\n\nimport csv\nfrom datetime import date, datetime\nfrom pathlib import Path\nfrom zoneinfo import ZoneInfo\n\nfrom ml.forecast.core import grid, parse_key, rounded\n\nFIELDS = [\'route\', \'date\', \'hour\', \'prediction\']\nSTART, END = date(2025, 11, 1), date(2025, 12, 31)\n\n\ndef read_sample(path):\n    with Path(path).open(encoding=\'utf-8-sig\', newline=\'\') as f:\n        reader = csv.DictReader(f, delimiter=\';\')\n        if reader.fieldnames != FIELDS:\n            raise ValueError(f\'Unexpected sample schema: {reader.fieldnames}\')\n        rows = list(reader)\n    if any(None in r or any(v is None for v in r.values()) for r in rows):\n        raise ValueError(\'Malformed sample row\')\n    keys = [parse_key(r) for r in rows]\n    routes = sorted({k[0] for k in keys}, key=int)\n    if len(keys) != 14640 or len(set(keys)) != len(keys):\n        raise ValueError(\'Sample must contain exactly 14640 unique keys\')\n    if len(routes) != 10 or \'5\' not in routes:\n        raise ValueError(\'Sample must contain ten routes, including route 5\')\n    if set(keys) != set(grid(routes, START, END)):\n        raise ValueError(\'Sample must cover every hour of November–December 2025\')\n    return rows, keys\n\n\ndef validate(sample_path, output_path):\n    rows, keys = read_sample(sample_path)\n    other, other_keys = read_sample(output_path)\n    if keys != other_keys:\n        raise ValueError(\'Submission changed sample ordering\')\n    for expected, actual in zip(rows, other):\n        if any(expected[k] != actual[k] for k in FIELDS[:-1]):\n            raise ValueError(\'Non-target sample text changed\')\n        value = actual[\'prediction\']\n        if not value.isascii() or not value.isdecimal() or str(int(value)) != value:\n            raise ValueError(\'Submission predictions must be canonical nonnegative integers\')\n        rounded(int(value))\n    return {\'rows\': len(keys), \'routes\': len({k[0] for k in keys}),\n            \'route5_rows\': sum(k[0] == \'5\' for k in keys), \'prediction_sum\':\n            sum(int(row[\'prediction\']) for row in other)}\n\n\ndef write_submission(sample_path, predictions, output_path):\n    if Path(sample_path).resolve() == Path(output_path).resolve():\n        raise ValueError(\'Never overwrite the original sample\')\n    rows, keys = read_sample(sample_path)\n    if set(predictions) != set(keys):\n        raise ValueError(\'Prediction keys must match sample exactly; no missing/extra rows\')\n    output_path = Path(output_path)\n    output_path.parent.mkdir(parents=True, exist_ok=True)\n    values = [rounded(predictions[k]) for k in keys]\n    with output_path.open(\'w\', encoding=\'utf-8\', newline=\'\') as f:\n        writer = csv.DictWriter(f, fieldnames=FIELDS, delimiter=\';\', lineterminator=\'\\n\')\n        writer.writeheader()\n        for row, value in zip(rows, values):\n            writer.writerow({**row, \'prediction\': value})\n    return validate(sample_path, output_path)\n\n\ndef export_api(keys, predictions, output_path, model_version):\n    if len(set(keys)) != len(keys) or set(keys) != set(predictions):\n        raise ValueError(\'API export key mismatch\')\n    with Path(output_path).open(\'w\', encoding=\'utf-8\', newline=\'\') as f:\n        writer = csv.writer(f, lineterminator=\'\\n\')\n        writer.writerow([\'route_id\', \'timestamp_msk\', \'pred_validations\',\n                         \'model_version\', \'source_mode\'])\n        for r, d, h in keys:\n            stamp = datetime(d.year, d.month, d.day, h, tzinfo=ZoneInfo(\'Europe/Moscow\'))\n            writer.writerow([r, stamp.isoformat(), rounded(predictions[r, d, h]),\n                             model_version, \'as_of_2025_10_31\'])\n\n\nif __name__ == \'__main__\':\n    import argparse\n    import json\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument(\'--sample\', required=True)\n    parser.add_argument(\'--submission\', required=True)\n    args = parser.parse_args()\n    print(json.dumps(validate(args.sample, args.submission), ensure_ascii=False, indent=2))\n'}
SOURCE_HASHES = {'ml/forecast/core.py': '819cfdc88d5b6b96bf80fb0e019eeef0678628b617d2dd077409d26265369b5f', 'ml/forecast/models.py': '0f382e23aa359aa6d5c8597f7423715852327454fe3083b647270e3c226f4c89', 'ml/forecast/run.py': 'bd767e33e9eba7cac4ac41aa24be7b96b641d30af2572570ae28a933d9878e8d', 'ml/forecast/prepare_inputs.py': '9fdb609444a3e443961504e76bda2ccb1d02dadb3ad88bb0420f70fe01711dd5', 'ml/forecast/test_pipeline.py': '267a014e6b267f4c792b7a90c8679c98ee94e3e36e070baa706a89f5fa8f399b', 'ml/forecast/download_yandex.py': '300dec623de4bf983a7960e23bbd85a128c120d40e25059d1a000d898c2ff37d', 'ml/forecast/test_download_yandex.py': '9ca4e25061931b50b2b1730e4085f44add1c9d9a0071cd1290779e5b939666d1', 'ml/submit/adapter.py': '9767bfd9573ea7637669e37bac42e75e45205f29d533d24d1d397f646047d97d'}
for name, source in SOURCES.items():
    if hashlib.sha256(source.encode()).hexdigest() != SOURCE_HASHES[name]:
        raise RuntimeError('Изменился встроенный исходник: ' + name)
    target = RUNTIME / name
    target.parent.mkdir(parents=True, exist_ok=True)
    target.write_text(source, encoding='utf-8')
print('Развёрнуты и проверены', len(SOURCES), 'модулей DS‑1.')

def run_module(module, *arguments):
    command = [sys.executable, '-u', '-m', module, *map(str, arguments)]
    with subprocess.Popen(command, cwd=RUNTIME, env=os.environ.copy(),
                          stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                          text=True, bufsize=1) as process:
        for line in process.stdout:
            print(line, end='', flush=True)
        result = process.wait()
    if result:
        raise RuntimeError(f'{module} завершился с ошибкой {result}. См. вывод выше.')

if RUN_TESTS:
    run_module('unittest', 'discover', '-s', 'ml/forecast', '-p', 'test_*.py', '-v')


## 3. Скачать архив с Яндекс Диска и найти входы
Официальный публичный API не требует токена. Скачивание идёт потоком в `/tmp`,
с прогрессом, повторными попытками, проверкой размера и хеша. При повторном запуске
проверенный архив используется снова; частичная загрузка докачивается, если сервер
поддерживает Range. В память целый архив не загружается.

Распаковываются только небольшие метки и шаблон; сырые транзакции остаются в ZIP.
При сетевой ошибке ячейка остановится с объяснением, а не начнёт обучение без данных.
Официальный способ получения ссылки: https://yandex.cloud/en/docs/datasphere/operations/data/connect-to-ya-disk


In [ ]:
SEARCH_ROOT = INPUT_ROOT
if DATA_SOURCE == 'yandex':
    import importlib.util
    spec = importlib.util.spec_from_file_location('ds1_yandex', RUNTIME / 'ml/forecast/download_yandex.py')
    downloader = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(downloader)
    downloaded_archive = downloader.download(YANDEX_URL, DOWNLOAD_CACHE)
    SEARCH_ROOT = downloaded_archive.parent
elif DATA_SOURCE != 'input':
    raise ValueError("DATA_SOURCE должен быть 'yandex' или 'input'")
if not SEARCH_ROOT.is_dir():
    raise FileNotFoundError('Нет входной папки. Для ручного режима добавьте набор через Add Input.')
required = {'train': 'labels_day_train.csv', 'test': 'labels_day_test.csv',
            'sample': 'test_submission.csv'}
files = {}
for role, name in required.items():
    if INPUT_FILES[role]:
        path = Path(INPUT_FILES[role])
        if not path.is_file():
            raise FileNotFoundError(path)
        files[role] = path
    else:
        matches = sorted(SEARCH_ROOT.rglob(name))
        if len(matches) > 1:
            raise RuntimeError(f'Несколько {name}: {matches}. Укажите INPUT_FILES[{role!r}].')
        if matches:
            files[role] = matches[0]

missing = set(required) - set(files)
archive_candidates = {role: [] for role in missing}
if missing:
    for archive in sorted(SEARCH_ROOT.rglob('*.zip')):
        with zipfile.ZipFile(archive) as bundle:
            for info in bundle.infolist():
                for role in missing:
                    if Path(info.filename).name == required[role]:
                        archive_candidates[role].append((archive, info.filename, info.file_size))
    for role in sorted(missing):
        matches = archive_candidates[role]
        if len(matches) != 1:
            raise RuntimeError(f'{required[role]}: найдено вариантов в ZIP: {len(matches)}. '
                               'Добавьте этот файл отдельно и укажите путь в INPUT_FILES.')
        archive, member, size = matches[0]
        if size > 20_000_000:
            raise ValueError('Неожиданно большой файл меток/шаблона: ' + member)
        with zipfile.ZipFile(archive) as bundle:
            path = EXTRACTED / required[role]
            path.write_bytes(bundle.read(member))
            files[role] = path

for role, path in files.items():
    print(role, '→', path, '|', f'{path.stat().st_size:,}', 'байт')
# Описание внутри исходного ZIP тоже можно прочитать без извлечения raw CSV.
for archive in sorted(SEARCH_ROOT.rglob('*.zip')):
    with zipfile.ZipFile(archive) as bundle:
        descriptions_in_zip = [i for i in bundle.infolist()
                               if Path(i.filename).name.lower() == 'readme.md' and i.file_size < 200_000]
        if len(descriptions_in_zip) == 1:
            description_path = EXTRACTED / 'README.md'
            description_path.write_bytes(bundle.read(descriptions_in_zip[0]))
            print('Описание из архива:\n', description_path.read_text(encoding='utf-8-sig'))
descriptions = sorted(SEARCH_ROOT.rglob('README.md'))
if len(descriptions) == 1 and descriptions[0].stat().st_size < 200_000:
    print('\nОписание набора:\n', descriptions[0].read_text(encoding='utf-8-sig'))
else:
    print('Описание не показано автоматически; сверьте README организаторов отдельно.')


## 4. Обучение и временная проверка

Все семь кандидатов сравниваются на мае–июне и июле–августе. Затем выбранный
кандидат и два baseline проверяются на сентябре–октябре. Если кандидат хуже
контролей, остаётся baseline. Финальная модель переобучается на январе–октябре.

Логи показывают WAPE-score: **выше лучше**. Результат сентября–октября участвует
в выборе модели и не является независимым тестом. Ноябрь–декабрь прогнозируется
без новых фактов; в рекурсивной модели лаги обновляются предсказаниями.


In [ ]:
arguments = ['--train', files['train'], '--test', files['test'],
             '--sample', files['sample'], '--out', OUT]
if PROFILES_ONLY:
    arguments.append('--profiles-only')
# Используется только при локальной проверке самого ноутбука; на Kaggle не задавать.
if os.environ.get('MTHACK_NOTEBOOK_TEST') == 'synthetic':
    arguments.append('--synthetic')
run_module('ml.forecast.run', *arguments)


## 5. Результаты проверки
Сводные метрики и метрики по маршрутам; score платформы станет известен только после подачи CSV.


In [ ]:
from IPython.display import display, Markdown, FileLink
report = json.loads((OUT / 'report.json').read_text())
lines = ['| Период | Модель | WAPE-score | Строк | Пропущено меток |',
         '|---|---|---:|---:|---:|']
for period, models in report['folds'].items():
    for name, result in models.items():
        metric = result['metrics']['all']
        lines.append(f"| {period} | {name} | {metric['wape_score']:.6f} | "
                     f"{metric['rows']} | {metric['missing_label_rows']} |")
display(Markdown('\n'.join(lines)))
chosen = report['selection']['selected']
print('Выбрана модель:', chosen)
print('Версия:', report['model_version'])
print('Cold start:', report['cold_start'])
lines = ['| Маршрут, сентябрь–октябрь | WAPE-score | Сумма факта | Сумма абсолютных ошибок |',
         '|---|---:|---:|---:|']
for group, metric in report['folds']['sep_oct'][chosen]['metrics'].items():
    if group.startswith('route:'):
        value = metric['wape_score']
        text = f'{value:.6f}' if value is not None else 'не определён'
        lines.append(f"| {group[6:]} | {text} | {metric['actual_sum']} | {metric['absolute_error_sum']} |")
display(Markdown('\n'.join(lines)))
display(Markdown('**Неизвестный ноябрь–декабрь:** фактическая метрика пока отсутствует. '
                 'Праздники/переносы дней, погода и проверенное расписание пока не включены.'))


## 6. Скачать результат

Повторно проверяем CSV. Готовые файлы появятся также в **Output**:
- `submission.csv` — файл для конкурсной платформы;
- `report.json` — пришлите его DS‑1 для разбора метрик;
- `forecast_api.csv` — почасовой прогноз для fullstack;
- `ds1_results/model.local.pkl` — локальный обученный артефакт, не публиковать.

Ноутбук сам ничего не отправляет на платформу. Сохраните версию с результатами,
чтобы не потерять файлы после завершения сессии.


In [ ]:
import shutil
run_module('ml.submit.adapter', '--sample', files['sample'],
           '--submission', OUT / 'submission.csv')
is_synthetic = report['data_kind'] == 'synthetic_not_for_submission'
if is_synthetic:
    print('ВНИМАНИЕ: искусственные данные. Этот CSV НЕ подавать на платформу.')
for name in ['submission.csv', 'report.json', 'forecast_api.csv']:
    target = WORK / name
    shutil.copy2(OUT / name, target)
    print(target.name, '|', f'{target.stat().st_size:,}', 'байт')
    # В Kaggle ссылки ведут на файлы относительно рабочей папки ноутбука.
    display(FileLink(str(target.relative_to(Path.cwd())) if target.is_relative_to(Path.cwd())
                     else str(target)))
with (WORK / 'submission.csv').open(encoding='utf-8', newline='') as f:
    preview = csv.reader(f, delimiter=';')
    for _, row in zip(range(6), preview):
        print(';'.join(row))
print('Готово. Пришлите report.json для оценки реального качества.')


### Воспроизводимость
В `report.json` сохранены SHA256 входов, исходников и выходов, версии библиотек,
параметры моделей и правила выбора. Пакеты Kaggle не обновляются автоматически:
используется установленная среда, её версии записываются в отчёт.
Исходники: `Dianka678/mthack-tram`, ветка `ds1/forecast-submission`, PR #3.
Официальная справка Kaggle: https://www.kaggle.com/docs/notebooks
